# Qwen3-Reranker-0.6B — DIMER query-document reranking tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-reranker-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-reranker-pipeline/blob/main/tutorials/qwen3_reranker_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Reranker--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B)
[![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding)
[![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** pointwise query-document relevance reranking using the pinned Qwen3-Reranker-0.6B weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`Qwen3RerankerPipeline`) rather than reimplementing model inference. At inference each `(query, document)` pair is wrapped in the fixed upstream system/user/assistant prompt together with an instruction, one forward pass of the causal language model reads the last-position logits of the `yes` and `no` tokens, and a two-way softmax turns them into a **relevance score** in [0, 1]: the `yes` share. **The score is not a calibrated probability**, no threshold is shipped, and the `ranking` the pipeline returns is an ordering of the supplied pairs, not an acceptance decision — the caller owns any cut-off. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the model, tokenizer and prompt convention; what this repository adds is manifest verification, input validation and ceilings, prompt assembly, the two-logit read-out, and a fixed output contract. Free-text generation is deliberately not reachable through this package.

**Learning objectives:** bootstrap the repository in a fresh runtime, author a synthetic query and candidate set (or upload your own), surface the pipeline's ceilings, stage and digest-verify the immutable upstream snapshot, rerank through the public API, read scores and ranking correctly, understand why no metric is reported and what labelled judgements a real evaluation needs, and export the ranking with identifiers plus provenance.

**This notebook does not demonstrate:** embedding or retrieval over a corpus (the Qwen3-Embedding sibling covers first-stage retrieval), text generation, listwise or pairwise comparison between documents, multilingual quality claims, or any calibrated relevance threshold. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16 there, the checkpoint's native dtype); the model card's CPU smoke scored four short pairs in about 1 s after a 5 s load, so the three-pair default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.19 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python; what a softmax over two logits is and why it is not a calibrated probability; what a relevance judgement is.
- **Data:** the default sample is a synthetic query and three candidate passages authored in code; BYOD is one UTF-8 text file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `huggingface-hub`, `safetensors`, `numpy`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CPU and bfloat16 on CUDA (the checkpoint's native `torch_dtype`), so scores differ slightly between the two device paths; no compilation or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/qwen3-reranker-pipeline.git'
REPO_NAME = 'qwen3-reranker-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: one query and three candidate passages written in this cell — one that answers the query, one on the same topic that does not answer it, and one unrelated — so it needs no download and contains no personal data. It ships **no relevance judgements** beyond the author's intent, so the scores it produces are smoke/sanity evidence that the code path works (the answering passage is expected to outrank the unrelated one), never a retrieval-quality measurement and never benchmark evidence.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose first non-empty line is the query and whose remaining non-empty lines are candidate documents (one per line, at most `MAX_PAIRS` documents, each at most `MAX_TEXT_CHARS` characters); every document is paired with the query. The upload stays inside this runtime. If you also hold relevance judgements for your candidates, keep them outside the notebook — Section 5 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    lines = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if len(lines) < 2:
        raise ValueError(f'{sample_name}: expected a query line followed by at least one document line')
    query, documents = lines[0], lines[1:]
    sample_kind = 'BYOD upload'
else:
    query = 'What is the capital of China?'
    documents = [
        'The capital of China is Beijing, which has been the seat of government since 1949.',
        'Shanghai is the largest city in China by population and a major financial centre.',
        'Gravity is the force by which a planet or other body draws objects toward its centre.',
    ]
    sample_name = 'synthetic_capital_query'
    sample_kind = 'synthetic (authored in this cell)'
doc_ids = [f'doc{index:02d}' for index in range(len(documents))]
pairs = [(query, document) for document in documents]
sample_sha256 = hashlib.sha256('\n'.join([query, *documents]).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'query': query, 'documents': len(documents), 'text_sha256': sample_sha256})
for doc_id, document in zip(doc_ids, documents):
    print(f'{doc_id}: {document[:100]}')

## 3. Validate the pairs against the pipeline ceilings

The pipeline enforces three operational ceilings, imported here from the package so the values shown are the ones in force: `MAX_PAIRS` (pairs per `rerank` call), `MAX_TEXT_CHARS` (characters per query or document, checked before tokenisation) and `MAX_TEXT_TOKENS` (total prompt tokens including the fixed prefix/suffix; longer prompts are truncated longest-first and flagged per pair in `truncated`). The default `instruction` is also imported and shown — it is part of the prompt and changes the scores, so a deployment must fix it deliberately. This cell surfaces the ceilings and checks the batch before any model work, naming the failing condition and the corrective action; `rerank()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. The notebook does not trim or alter the texts; any token-level truncation happens inside the pipeline and is reported after the call.

In [ ]:
from qwen3_reranker_pipeline import DEFAULT_INSTRUCTION, MAX_PAIRS, MAX_TEXT_CHARS, MAX_TEXT_TOKENS

ceilings = {'MAX_PAIRS': MAX_PAIRS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS}
instruction = DEFAULT_INSTRUCTION
print(ceilings)
print({'instruction': instruction})
problems = []
if not 1 <= len(pairs) <= MAX_PAIRS:
    problems.append(f'{len(pairs)} pairs is outside 1..MAX_PAIRS={MAX_PAIRS}: supply fewer documents or split the batch')
for doc_id, (pair_query, document) in zip(doc_ids, pairs):
    for name, text in (('query', pair_query), (doc_id, document)):
        if not text.strip():
            problems.append(f'{name} is empty: remove blank lines from the input')
        if len(text) > MAX_TEXT_CHARS:
            problems.append(f'{name} has {len(text)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten the text')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'pairs': len(pairs), 'longest_chars': max(len(text) for pair in pairs for text in pair), 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and refuses remote model code (`trust_remote_code=False`). The repository commits the DIMER snapshot manifest (`weights/qwen3-reranker-0.6b/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the 13 snapshot files) and the small config/tokenizer files, but git-ignores the 1.19 GB `model.safetensors`, so a fresh clone must stage the missing file first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot()` then re-hashes every listed file and raises on the first size or digest mismatch, and only afterwards does `from_pretrained` load tokenizer and model from that verified directory with `local_files_only=True`, checking that the tokenizer maps `yes`/`no` to the expected token ids. The effective model identity and the selected device are printed before inference.

In [ ]:
from qwen3_reranker_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, Qwen3RerankerPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': str(WEIGHTS_DIR), 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')})
pipe = Qwen3RerankerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'dtype': 'bfloat16' if pipe.device.startswith('cuda') else 'float32'})

## 5. Rerank and interpret the scores

`rerank(pairs, instruction=...)` returns `scores` (one float in [0, 1] per pair, **aligned with the input order**), `ranking` (pair indices sorted by descending score, stable), `score_kind`, the `instruction` used, `n_tokens` per prompt, `truncated` flags, and the model identity. **Score semantics:** each score is the softmax share of the `yes` logit against the `no` logit — a relevance score that orders candidates for one query; it is not a calibrated probability that the document is relevant, scores for different queries are not comparable as absolute values, and the pipeline ships no threshold. Any accept/reject cut-off (for example "show only candidates above 0.5") is owned by the caller and must be set on their own labelled pairs.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**. Ranking quality needs relevance judgements — for each query, which candidates are relevant (binary or graded) — from which the caller computes nDCG@k, MRR or precision@k with their own evaluation code, over enough queries to state a dispersion. The synthetic sample has no judgements, so **no metric is reported**; the sanity check below (the answering passage outranks the unrelated one) is a falsifiable plumbing check on one query, not a retrieval-quality result, and the upstream benchmark figures quoted in the model card are upstream claims, not measured here. Scores differ slightly between the CPU float32 and CUDA bfloat16 paths and can reorder near-tied candidates. The runtime figure is measured on the runtime identified in Section 1 for this batch and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.rerank(pairs, instruction=instruction)
elapsed = time.perf_counter() - started
scores = result['scores']
checks = {
    'one_score_per_pair': len(scores) == len(pairs),
    'scores_in_unit_interval': all(0.0 <= s <= 1.0 for s in scores),
    'ranking_is_permutation': sorted(result['ranking']) == list(range(len(pairs))),
    'nothing_truncated': not any(result['truncated']),
}
if not all(checks.values()):
    raise RuntimeError(f'rerank output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key not in ('scores', 'ranking')})
print({'seconds': round(elapsed, 3), 'checks': checks})
print(f"query: {query}")
for rank, index in enumerate(result['ranking'], start=1):
    print(f"{rank:>2}. {doc_ids[index]}  score {scores[index]:.4f}  tokens {result['n_tokens'][index]:>4}  {documents[index][:80]}")
metrics = {}
if not USE_BYOD:
    sanity = {'answering_outranks_unrelated': scores[0] > scores[2]}
    print({'sanity_check': sanity, 'note': 'falsifiable plumbing check on one synthetic query; not a metric'})
print('no metric is reported: the sample has no relevance judgements, and the repository ships no metric helper; compute nDCG/MRR on your own judged pairs')

## 6. Export the ranking and provenance

One JSON record is written under `outputs/`: an `items` list with, per pair, its identifier, document text, score, rank, token count and truncation flag (so every score maps back to its input), the query, the instruction, the `ranking`, the `score_kind`, the sanity checks, the ceilings in force, the empty metric block, the sample identity and digest, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, NumPy, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
rank_of = {index: rank for rank, index in enumerate(result['ranking'], start=1)}
payload = {
    'query': query,
    'instruction': result['instruction'],
    'items': [
        {'id': doc_ids[index], 'document': documents[index], 'score': scores[index], 'rank': rank_of[index], 'n_tokens': result['n_tokens'][index], 'truncated': result['truncated'][index]}
        for index in range(len(pairs))
    ],
    'ranking': [doc_ids[index] for index in result['ranking']],
    'score_kind': result['score_kind'],
    'sanity_checks': checks,
    'ceilings': ceilings,
    'metrics': metrics,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'seconds': round(elapsed, 3),
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'numpy': numpy.__version__,
        'device': pipe.device,
        'dtype': 'bfloat16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/qwen3_reranker_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/qwen3_reranker_result.json')

## Interpretation and limits

Each score is the `yes` share of a two-way softmax over the reranker's last-position logits: a relevance score that orders candidates for one query, not a calibrated probability, not comparable as an absolute value across queries or instructions, and never converted to a decision by the pipeline — the caller owns any threshold and must set it on their own judged pairs. On the synthetic sample the ordering is plumbing evidence only; no metric is reported because none can be computed without relevance judgements, and a real evaluation needs judged candidates for many queries and the caller's own nDCG/MRR code. The instruction is part of the prompt and changes the scores; prompts beyond `MAX_TEXT_TOKENS` are truncated longest-first and flagged; the pipeline exposes no generation, no listwise comparison, and no corpus retrieval. The forward pass is deterministic on a fixed device and dtype, but CPU (float32) and CUDA (bfloat16) scores differ slightly and can reorder near-tied candidates.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated pairs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, retrieval quality on any domain, a usable relevance threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/qwen3-reranker-0.6b/` and rerun Section 4. A `ValueError` naming `MAX_PAIRS` or `MAX_TEXT_CHARS` in Section 3: reduce or shorten the BYOD lines and rerun from Section 2. A `truncated` flag set to `True` in Section 5: that prompt exceeded `MAX_TEXT_TOKENS` and was cut longest-first — shorten the document if the cut matters.

**Next experiments.** Upload a query with a dozen candidates you can judge yourself and compare the pipeline's ranking with your judgements; change `instruction` in Section 3 to a task-specific one and observe how the scores move; run the same batch on a CUDA runtime and compare the bfloat16 scores with the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/Qwen/Qwen3-Reranker-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding technical report: https://arxiv.org/abs/2506.05176